# Input files for KUL-TN-21

### Setup notebook

In [4]:
# Alow changes to the PlatoSim code outside this notebook
%load_ext autoreload
%autoreload 2

# Configure figure in notebook
%matplotlib notebook

### Imports

In [5]:
# Second part libraries
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter

# PlatoSim libraries
from platosim.simfile import SimFile
from platosim.plot import plotPhotometry
from platosim.utilities import errorcode, normalize
from platosim.lightcurve import LightCurve
from platosim.matplotlibrc import setup_notebook
setup_notebook()

---
## Create input misalignment models
---

In [6]:
# Choose directory to save and unpack data
idir = '/lhome/nicholas/software/workdir/jitterImpact/input/'

# Constants
ppmh = 144   # For a cadence of 25s
model = 'y ~ x'
ms, aa = 3, 0.1

# Perfect pointing to provisional SPF (ICRS - equatorial)
ra  = 86.79870508
dec = -46.39594703
rot = -4.0
ICRS = np.array([ra, dec, rot])
quarters = np.array([1, 2, 3, 4, 5, 6, 7, 8])
sigma = 3

### Generate PRE file

### Generate APE file

---
## Microscanning
---

In [ ]:
# Load file
filename = os.getenv('PLATO_WORKDIR') + '/input/spiral_8Hz_airbus_3h.dat'
filename = os.getenv('PLATO_WORKDIR') + '/input/spiral_8Hz_airbus_3h.dat'

# filename = os.getenv('PLATO_PROJECT_HOME') + '/inputfiles/spiral_8Hz_airbus_3h.txt'
# data = pd.read_csv(filename, sep=' ', names=["t", "x", "y", "z"], header=None)
data = np.loadtxt(filename)
t, x, y, z = data[:,0], data[:,1], data[:,2], data[:,3]

cadence  = 25.  # [s]                                                                                                                                                                             
quarter  = 0                                                                                                                                                                                           
day = 86400.                                                                                                                                                                                            
time_points = int(t[-1] / cadence)
timeStart = round(90. * quarter * day)                                                                                                                                                                   
timeEnd   = round(timeStart + time_points * cadence)                                                                                                                                                    
time = np.arange(timeStart, timeEnd, cadence)

# Find points at cadence
dex = [ut.findNearestIndex(t, time[i]) for i in range(len(time))]


## Correct pandas columns

In [7]:
# # User parameters
# idir = "/STER/platodata/PLATOSIM/simulations_PLATO-PL-KUL-TN-0020/P1"
# odir = "/STER/platodata/PLATOSIM/simulations_PLATO-PL-KUL-TN-0020/P1_corrected"
# # Correct columns
# phot = LightCurve(idir, mode="multi")
# phot.correct_and_save(idir, odir, numBegin=818, numEnd=1000)

In [90]:
# idir = "/STER/platodata/PLATOSIM/simulations_PLATO-PL-KUL-TN-0020/P1"
# ofile = os.getcwd() + "/bad_pandas.txt"
# phot = LightCurve(idir, mode="multi")
# phot.bad_files(ofile, numBegin=3, numEnd=4)

---
## Download and plot light curve
---

In [7]:
idir = "/STER/platodata/PLATOSIM/simulations_PLATO-PL-KUL-TN-0020/P1_corrected/000000004"
phot = LightCurve(idir, mode="multi")

# To fetch a specific light curve use instead:
# lc = LightCurve(f"{idir}/000000083/000000083_Ncam1.1_Q23.ftr")

In [92]:
# Unzip all compressed files for the same star
phot.unpack()

In [8]:
# Fetch the first file
filenames = phot.files("ftr")
filenames[0]

IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
# Fetch light curve
lc = LightCurve(filenames[0])
lc.data().head()

In [ ]:
# Get target star information
star = lc.star_info(phot.files("cat")[0])
print(f"""
Pmag : {star[0]}
rOA  : {star[1]}
rCOB : {star[2]}
nCon : {star[3]}
rCon : {star[4]}
dMag : {star[5]}
SPR  : {star[6]}
""")

In [ ]:
# Get noise-less light curve
lc.varsource()

In [ ]:
# Plot the input noise-less light curve
fig, ax = lc.plot_varsource();

In [ ]:
# Plot the simulation, running median, and binned data
fig, ax = lc.plot(time_unit="d", binsize=5, median_filter=1);

In [ ]:
# Plot a quick O-C comparison plot 
fig, ax = lc.plot_oc();

In [ ]:
# Detrend light curve
lc.plot_detrend(poly_deg=4, binsize=1);

In [ ]:
# Get NSR per 1h for detrended light curve 
lc.detrend(poly_deg=3)
lc.getNSR(column="flux_det", binhour=1, influx="ppm")

In [ ]:
# With the long trend systematics the NSR is somewhat larger
lc.getNSR()

In [ ]:
# Remove files again to keep server clean
phot.remove()